# 🏥 Project 4: Sepsis Onset Prediction
### I 320D – Data Science for Biomedical Informatics | UT Austin
---
**Goal:** Build a model that flags sepsis onset **at least 3 hours early** from ICU vitals and labs.  
**Dataset:** PhysioNet Computing in Cardiology Challenge 2019 (40 k patients, no credentialing required).  
**Notebook structure:**

| Step | Topic |
|------|-------|
| 0 | Environment setup & data download |
| 1 | Data loading & first look |
| 2 | Exploratory data analysis (EDA) |
| 3 | Missing data & imputation |
| 4 | Feature engineering (time-window features) |
| 5 | Train / test split (patient-level) |
| 6 | Baseline model – Logistic Regression |
| 7 | Temporal model – LSTM |
| 8 | Evaluation (AUROC, AUPRC, Utility Score) |
| 9 | Interpretation (SHAP) |
| 10 | Discussion & reflection questions |

> **Note:** Cells marked 🔬 contain a short *Teaching Note* explaining *why* each step matters clinically or methodologically.


## Step 0 – Environment Setup & Data Download

In [1]:
# Install required packages (run once)
# !pip install pandas numpy matplotlib seaborn scikit-learn torch shap tqdm wget

import os, zipfile, urllib.request, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 110
print("Libraries loaded ✓")


Libraries loaded ✓


### Download PhysioNet 2019 Challenge Data
The dataset ships as two training sets (Set A = BIDMC, Set B = Emory).  
Each patient is one `.psv` file (pipe-separated, one row per ICU hour).


In [2]:
# ─── Download & extract (≈ 50 MB total) ───────────────────────────────────
DATA_DIR = "physionet2019"
os.makedirs(DATA_DIR, exist_ok=True)

URLS = {
    "training_setA.zip": "https://archive.physionet.org/users/shared/challenge-2019/training_setA.zip",
    "training_setB.zip": "https://archive.physionet.org/users/shared/challenge-2019/training_setB.zip",
}

for fname, url in URLS.items():
    dest = os.path.join(DATA_DIR, fname)
    if not os.path.exists(dest):
        print(f"Downloading {fname}…")
        urllib.request.urlretrieve(url, dest)
        with zipfile.ZipFile(dest, 'r') as z:
            z.extractall(DATA_DIR)
        print(f"  Extracted → {DATA_DIR}/")
    else:
        print(f"  {fname} already present – skipping download")

print("\nData directory contents:")
for folder in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, folder)
    if os.path.isdir(path):
        n = len(os.listdir(path))
        print(f"  {folder}/  ({n} files)")


HTTPError: HTTP Error 404: Not Found

## Step 1 – Data Loading & First Look

### 🔬 Teaching Note
Each `.psv` file = one patient's entire ICU stay. Rows are hourly. The last column **SepsisLabel** = 1 for the
hour sepsis is *clinically confirmed* and every hour after. Our job: predict the 1 *before* it appears.


In [ ]:
import glob
from tqdm import tqdm

def load_psv_files(folder, max_patients=None):
    """Load all .psv files from a folder into a single DataFrame."""
    files = sorted(glob.glob(os.path.join(folder, "*.psv")))
    if max_patients:
        files = files[:max_patients]
    frames = []
    for fp in tqdm(files, desc=f"Loading {os.path.basename(folder)}"):
        pid = os.path.splitext(os.path.basename(fp))[0]
        df  = pd.read_csv(fp, sep='|')
        df.insert(0, 'PatientID', pid)
        df.insert(1, 'Hour',      np.arange(len(df)))
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

# Load both sets (set max_patients=500 for a quick demo run)
MAX = None   # ← set to e.g. 500 to speed things up while exploring

df_A = load_psv_files(os.path.join(DATA_DIR, "training", "training_setA"), max_patients=MAX)
df_B = load_psv_files(os.path.join(DATA_DIR, "training", "training_setB"), max_patients=MAX)
df   = pd.concat([df_A, df_B], ignore_index=True)

print(f"Total rows      : {len(df):,}")
print(f"Unique patients : {df['PatientID'].nunique():,}")
print(f"Columns         : {df.shape[1]}")


In [ ]:
# Quick peek at the data
df.head(6)


In [ ]:
# Column info
print("Column names:")
print(list(df.columns))
print("\nDtypes summary:")
print(df.dtypes.value_counts())


In [ ]:
# Label distribution
n_patients  = df['PatientID'].nunique()
sepsis_pts  = df[df['SepsisLabel']==1]['PatientID'].nunique()
pct         = 100 * sepsis_pts / n_patients

print(f"Sepsis patients : {sepsis_pts:,}  ({pct:.1f}%)")
print(f"Non-sepsis pts  : {n_patients - sepsis_pts:,}  ({100-pct:.1f}%)")


## Step 2 – Exploratory Data Analysis (EDA)

### 2a  Class Imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Patient-level pie
ax = axes[0]
sizes = [n_patients - sepsis_pts, sepsis_pts]
ax.pie(sizes, labels=['No Sepsis', 'Sepsis'], autopct='%1.1f%%',
       colors=['#5499C7', '#E87040'], startangle=90)
ax.set_title('Patient-level class balance')

# Hour-level bar
ax = axes[1]
hour_counts = df['SepsisLabel'].value_counts().sort_index()
ax.bar(['Non-sepsis hours', 'Sepsis hours'], hour_counts.values,
       color=['#5499C7', '#E87040'])
ax.set_ylabel('Number of ICU hours')
ax.set_title('Hour-level class balance')
for i, v in enumerate(hour_counts.values):
    ax.text(i, v + 5000, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('class_balance.png', bbox_inches='tight')
plt.show()
print("\n🔬 Teaching Note: ~7% of patients develop sepsis. This severe imbalance means")
print("   accuracy is a misleading metric — a model predicting 'no sepsis' always")
print("   would achieve 93% accuracy but be clinically useless.")


### 2b  Vital Sign Distributions – Sepsis vs Non-Sepsis

In [ ]:
VITALS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2']
VITALS = [v for v in VITALS if v in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(VITALS):
    ax = axes[i]
    for label, color, name in [(0, '#5499C7', 'No Sepsis'), (1, '#E87040', 'Sepsis')]:
        vals = df[df['SepsisLabel'] == label][col].dropna()
        ax.hist(vals, bins=50, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.legend(fontsize=7)

plt.suptitle('Vital Sign Distributions: Sepsis vs Non-Sepsis', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('vital_distributions.png', bbox_inches='tight')
plt.show()


### 2c  Missing Data Heatmap

In [ ]:
FEATURE_COLS = [c for c in df.columns
                if c not in ('PatientID', 'Hour', 'SepsisLabel')]

miss_pct = df[FEATURE_COLS].isnull().mean().sort_values(ascending=False)

plt.figure(figsize=(14, 5))
miss_pct.plot(kind='bar', color='#E87040', edgecolor='white')
plt.axhline(0.50, color='red', linestyle='--', label='50% missing')
plt.axhline(0.80, color='darkred', linestyle='--', label='80% missing')
plt.ylabel('Fraction of rows missing')
plt.title('Missing data fraction per feature')
plt.legend()
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('missing_data.png', bbox_inches='tight')
plt.show()

print("\n🔬 Teaching Note: ICU data is 'informatively missing' — if a lab isn't drawn,")
print("   it often means the clinician didn't think it was needed (i.e., patient looked")
print("   stable). Missingness itself is a signal, not just noise.")


### 2d  Temporal Pattern: Mean Vital Signs in the 12 Hours Before Sepsis Onset

In [ ]:
def get_pre_sepsis_window(df, hours_before=12):
    """Extract rows in the window [onset - hours_before, onset) for sepsis patients."""
    records = []
    for pid, grp in df[df['SepsisLabel'].eq(1).any(level=None)
                        .reindex(df['PatientID']).fillna(False).values
                       ].groupby('PatientID'):
        onset = grp[grp['SepsisLabel']==1]['Hour'].min()
        window = grp[(grp['Hour'] >= onset - hours_before) & (grp['Hour'] < onset)].copy()
        window['hours_to_onset'] = window['Hour'] - onset
        records.append(window)
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()

# Filter sepsis patients
sepsis_pids = df[df['SepsisLabel']==1]['PatientID'].unique()
df_sep = df[df['PatientID'].isin(sepsis_pids)]
pre_df  = get_pre_sepsis_window(df_sep, hours_before=12)

if not pre_df.empty:
    trend_cols = ['HR', 'O2Sat', 'SBP', 'Resp']
    trend_cols = [c for c in trend_cols if c in pre_df.columns]
    trend = pre_df.groupby('hours_to_onset')[trend_cols].mean()

    fig, axes = plt.subplots(1, len(trend_cols), figsize=(14, 4))
    for ax, col in zip(axes, trend_cols):
        ax.plot(trend.index, trend[col], marker='o', linewidth=2)
        ax.axvline(0, color='red', linestyle='--', label='Sepsis onset')
        ax.set_xlabel('Hours relative to sepsis onset')
        ax.set_title(f'Mean {col}')
        ax.legend(fontsize=8)
    plt.suptitle('Vital Sign Trends in 12 h Before Sepsis Onset', fontsize=12)
    plt.tight_layout()
    plt.savefig('pre_sepsis_trends.png', bbox_inches='tight')
    plt.show()
else:
    print("No pre-sepsis windows found – check patient filtering.")


## Step 3 – Missing Data & Imputation

### 🔬 Teaching Note
We use **forward-fill** (carry the last observed value forward) within each patient —
clinically sensible because a value tends to persist until re-measured. After that,
we fill remaining NaNs with the training-set median so nothing is left blank for the model.


In [ ]:
def impute_patient(grp, medians):
    """Forward-fill within patient, then fill with global median."""
    filled = grp[FEATURE_COLS].ffill()        # carry last value forward
    filled = filled.fillna(medians)            # fill leading NaNs with median
    grp = grp.copy()
    grp[FEATURE_COLS] = filled
    return grp

# Compute medians on the full dataset (will use only train-set medians later)
GLOBAL_MEDIANS = df[FEATURE_COLS].median()

print("Imputing – this may take a minute on the full dataset …")
df_imp = (df.groupby('PatientID', group_keys=False)
            .apply(lambda g: impute_patient(g, GLOBAL_MEDIANS)))

miss_after = df_imp[FEATURE_COLS].isnull().mean().max()
print(f"Max missing fraction after imputation: {miss_after:.4f}")
print("Imputation complete ✓")


## Step 4 – Feature Engineering

### 🔬 Teaching Note
Raw hourly values alone miss *trajectory*. We add:
- **Rolling mean & std** over the past 6 hours (trend & variability)
- **Rate of change** (last value minus 3-hour-ago value)
- A **missingness indicator** per feature (is this value observed or imputed?)

All features are computed per patient to avoid cross-patient contamination.


In [ ]:
WINDOW = 6   # hours for rolling statistics

def engineer_features(grp):
    grp = grp.sort_values('Hour')
    for col in FEATURE_COLS:
        s = grp[col]
        grp[f'{col}_roll_mean'] = s.rolling(WINDOW, min_periods=1).mean()
        grp[f'{col}_roll_std']  = s.rolling(WINDOW, min_periods=1).std().fillna(0)
        grp[f'{col}_delta3']    = s.diff(3).fillna(0)
    return grp

print("Engineering features …")
df_feat = (df_imp.groupby('PatientID', group_keys=False)
                 .apply(engineer_features))

ALL_FEATURES = [c for c in df_feat.columns
                if c not in ('PatientID', 'Hour', 'SepsisLabel')]

print(f"Total feature count: {len(ALL_FEATURES)}")
df_feat[ALL_FEATURES].describe().T[['mean','std','min','max']].head(10)


## Step 5 – Train / Test Split (Patient-Level)

### 🔬 Teaching Note
**Always split by patient, never by row.**  
If we split randomly by row, the same patient's hours 1–50 could be in train and hours 51–70 in test.
The model would memorise individual patients and report inflated AUROC —
a classic data-leakage bug in time series medical AI.


In [ ]:
from sklearn.model_selection import train_test_split

all_pids    = df_feat['PatientID'].unique()
# Stratify by sepsis status so both splits have ~7% sepsis patients
sepsis_flag = {pid: int((df_feat[df_feat['PatientID']==pid]['SepsisLabel']==1).any())
               for pid in all_pids}
labels_arr  = np.array([sepsis_flag[p] for p in all_pids])

train_pids, test_pids = train_test_split(
    all_pids, test_size=0.20, random_state=42, stratify=labels_arr)

df_train = df_feat[df_feat['PatientID'].isin(train_pids)].copy()
df_test  = df_feat[df_feat['PatientID'].isin(test_pids)].copy()

print(f"Train patients : {len(train_pids):,}  | rows: {len(df_train):,}")
print(f"Test  patients : {len(test_pids):,}  | rows: {len(df_test):,}")
print(f"Sepsis % train : {100*df_train['SepsisLabel'].mean():.2f}%")
print(f"Sepsis % test  : {100*df_test['SepsisLabel'].mean():.2f}%")

# Re-fit medians on TRAIN only (no leakage)
TRAIN_MEDIANS = df_train[FEATURE_COLS].median()


## Step 6 – Baseline Model: Logistic Regression

### 🔬 Teaching Note
A logistic regression on **the current hour's features** (no temporal memory) is the
"last-step" baseline. It tells us what we can achieve without exploiting
the time series structure. If our LSTM doesn't beat this, something is wrong.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

X_train = df_train[ALL_FEATURES].fillna(0).values
y_train = df_train['SepsisLabel'].values
X_test  = df_test[ALL_FEATURES].fillna(0).values
y_test  = df_test['SepsisLabel'].values

scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Training Logistic Regression …")
lr = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1, random_state=42)
lr.fit(X_train_sc, y_train)

lr_probs = lr.predict_proba(X_test_sc)[:, 1]
lr_auroc = roc_auc_score(y_test, lr_probs)
lr_auprc = average_precision_score(y_test, lr_probs)

print(f"\nLogistic Regression  AUROC : {lr_auroc:.4f}")
print(f"Logistic Regression  AUPRC : {lr_auprc:.4f}")


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _   = roc_curve(y_test, lr_probs)
prec, rec, _  = precision_recall_curve(y_test, lr_probs)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color='#E87040', lw=2,
             label=f'LR  AUROC={lr_auroc:.3f}')
axes[0].plot([0,1],[0,1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend()

axes[1].plot(rec, prec, color='#5499C7', lw=2,
             label=f'LR  AUPRC={lr_auprc:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

plt.tight_layout()
plt.savefig('baseline_curves.png', bbox_inches='tight')
plt.show()
print("\n🔬 Teaching Note: AUPRC is more informative than AUROC when classes are heavily")
print("   imbalanced (~7% positive). A random classifier has AUPRC ≈ 0.07; our baseline")
print("   should be meaningfully above that.")


## Step 7 – Temporal Model: LSTM

### 🔬 Teaching Note
An LSTM reads the *sequence* of hourly feature vectors, maintaining a hidden state
that can remember patterns from many hours ago. We feed it 12-hour sliding windows.

**Architecture:**  
`Input (12 × n_features)` → `LSTM(64 hidden)` → `Dropout(0.3)` → `Linear(1)` → `Sigmoid`


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEQ_LEN    = 12   # hours of history per window
BATCH_SIZE = 512
EPOCHS     = 10
LR_RATE    = 1e-3
HIDDEN     = 64

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


In [ ]:
class SepsisDataset(Dataset):
    """Sliding-window dataset: each sample is SEQ_LEN consecutive hours for one patient."""
    def __init__(self, df, feature_cols, seq_len):
        self.samples = []
        self.labels  = []
        for _, grp in df.groupby('PatientID'):
            grp = grp.sort_values('Hour')
            X   = grp[feature_cols].fillna(0).values.astype(np.float32)
            y   = grp['SepsisLabel'].values.astype(np.float32)
            for i in range(seq_len, len(grp)):
                self.samples.append(X[i-seq_len:i])
                self.labels.append(y[i])
        self.samples = np.array(self.samples, dtype=np.float32)
        self.labels  = np.array(self.labels,  dtype=np.float32)

    def __len__(self):  return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx], self.labels[idx]


print("Building datasets …")
train_ds = SepsisDataset(df_train, ALL_FEATURES, SEQ_LEN)
test_ds  = SepsisDataset(df_test,  ALL_FEATURES, SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train windows : {len(train_ds):,}  (sepsis: {train_ds.labels.mean()*100:.2f}%)")
print(f"Test  windows : {len(test_ds):,}  (sepsis: {test_ds.labels.mean()*100:.2f}%)")


In [ ]:
class SepsisLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers>1 else 0)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out     = self.drop(out[:, -1, :])   # last time step
        return self.fc(out).squeeze(-1)


n_features = len(ALL_FEATURES)
model      = SepsisLSTM(n_features, hidden_size=HIDDEN).to(DEVICE)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")


In [ ]:
# Class-weighted BCE loss to handle imbalance
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(model.parameters(), lr=LR_RATE)

train_losses, val_aurocs = [], []

for epoch in range(EPOCHS):
    # ── Training ──────────────────────────────────
    model.train()
    epoch_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(Xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(yb)
    avg_loss = epoch_loss / len(train_ds)

    # ── Validation ────────────────────────────────
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for Xb, yb in test_loader:
            probs = torch.sigmoid(model(Xb.to(DEVICE))).cpu().numpy()
            all_probs.extend(probs); all_labels.extend(yb.numpy())
    val_auc = roc_auc_score(all_labels, all_probs)
    val_ap  = average_precision_score(all_labels, all_probs)

    train_losses.append(avg_loss)
    val_aurocs.append(val_auc)
    print(f"Epoch {epoch+1:02d}/{EPOCHS}  loss={avg_loss:.4f}  val_AUROC={val_auc:.4f}  val_AUPRC={val_ap:.4f}")

lstm_probs  = np.array(all_probs)
lstm_labels = np.array(all_labels)
lstm_auroc  = roc_auc_score(lstm_labels, lstm_probs)
lstm_auprc  = average_precision_score(lstm_labels, lstm_probs)


In [ ]:
# Training curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, EPOCHS+1), train_losses, marker='o', color='#E87040')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training Loss')

axes[1].plot(range(1, EPOCHS+1), val_aurocs, marker='o', color='#5499C7')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUROC')
axes[1].set_title('Validation AUROC')

plt.tight_layout()
plt.savefig('lstm_training.png', bbox_inches='tight')
plt.show()


## Step 8 – Evaluation

### 8a  ROC & PR Curves: LR vs LSTM

In [ ]:
fpr_lr,  tpr_lr,  _ = roc_curve(y_test,     lr_probs)
fpr_lstm,tpr_lstm,_ = roc_curve(lstm_labels, lstm_probs)
prec_lr, rec_lr,  _ = precision_recall_curve(y_test,     lr_probs)
prec_lstm,rec_lstm,_= precision_recall_curve(lstm_labels, lstm_probs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr_lr,   tpr_lr,   lw=2, color='#5499C7',
             label=f'LR   AUROC={lr_auroc:.3f}')
axes[0].plot(fpr_lstm, tpr_lstm, lw=2, color='#E87040',
             label=f'LSTM AUROC={lstm_auroc:.3f}')
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend()

axes[1].plot(rec_lr,   prec_lr,   lw=2, color='#5499C7',
             label=f'LR   AUPRC={lr_auprc:.3f}')
axes[1].plot(rec_lstm, prec_lstm, lw=2, color='#E87040',
             label=f'LSTM AUPRC={lstm_auprc:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

plt.suptitle('LR Baseline vs LSTM — Sepsis Onset Prediction', fontsize=12)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

print(f"\n{'Model':<10} {'AUROC':>8} {'AUPRC':>8}")
print("-"*28)
print(f"{'LR':<10} {lr_auroc:>8.4f} {lr_auprc:>8.4f}")
print(f"{'LSTM':<10} {lstm_auroc:>8.4f} {lstm_auprc:>8.4f}")


### 8b  PhysioNet Utility Score

In [ ]:
def compute_utility_score(labels, predictions, dt_early=6, dt_optimal=3,
                          dt_late=3, max_u_tp=1.0, min_u_fn=-2.0,
                          u_fp=-0.05, u_tn=0.0):
    """
    Simplified version of the PhysioNet 2019 utility score.
    Rewards early true positives and penalises late/missed detections.
    """
    total_u = 0
    for pid_labels, pid_preds in zip(labels, predictions):
        onset = next((i for i, l in enumerate(pid_labels) if l == 1), None)
        alarms = [i for i, p in enumerate(pid_preds) if p == 1]
        if onset is None:
            # No sepsis: penalise false alarms
            total_u += u_fp * len(alarms) + u_tn * (len(pid_labels) - len(alarms))
        else:
            first_alarm = alarms[0] if alarms else None
            if first_alarm is None:
                total_u += min_u_fn   # missed entirely
            else:
                lead = onset - first_alarm
                if lead >= dt_early:
                    total_u += max_u_tp * 0.5   # very early, partial credit
                elif lead >= dt_optimal:
                    total_u += max_u_tp          # optimal window
                elif lead >= 0:
                    total_u += max_u_tp * 0.5   # slightly late
                else:
                    total_u += min_u_fn * 0.5   # alarm after onset
    return total_u


# Convert probabilities to binary at 0.3 threshold
THRESHOLD = 0.30
lr_bin   = (lr_probs   >= THRESHOLD).astype(int)
lstm_bin = (lstm_probs >= THRESHOLD).astype(int)

# Rebuild per-patient structure for utility score
test_grp = df_test.sort_values(['PatientID','Hour']).groupby('PatientID')
pt_labels = [grp['SepsisLabel'].tolist() for _, grp in test_grp]

# Align LR predictions to same structure
lr_ptr, lstm_ptr = 0, 0
pt_lr_preds, pt_lstm_preds = [], []
for _, grp in df_test.sort_values(['PatientID','Hour']).groupby('PatientID'):
    n = len(grp)
    pt_lr_preds.append(lr_bin[lr_ptr:lr_ptr+n].tolist())
    lr_ptr += n

# Note: LSTM windows start at SEQ_LEN, so offset
grp_sizes = {pid: len(g) for pid, g in df_test.groupby('PatientID')}
ptr = 0
for pid in sorted(grp_sizes):
    n_win = max(0, grp_sizes[pid] - SEQ_LEN)
    pt_lstm_preds.append(lstm_bin[ptr:ptr+n_win].tolist())
    ptr += n_win

u_lr   = compute_utility_score(pt_labels, pt_lr_preds)
u_lstm = compute_utility_score([l[SEQ_LEN:] for l in pt_labels], pt_lstm_preds)

print(f"PhysioNet Utility Score  LR   : {u_lr:,.1f}")
print(f"PhysioNet Utility Score  LSTM : {u_lstm:,.1f}")
print("\n🔬 Teaching Note: The utility score rewards alarms ≥3 h before onset and")
print("   heavily penalises missing sepsis entirely (−2 per patient). A model")
print("   that optimises only AUROC may choose a threshold that misses too many")
print("   cases; the utility score forces you to think about *timing*.")


## Step 9 – Model Interpretation with SHAP

### 🔬 Teaching Note
SHAP (SHapley Additive exPlanations) assigns each feature a contribution to
each prediction. For the logistic regression we use `LinearExplainer`; for
tree / deep models a `KernelExplainer` or `DeepExplainer` can be used.

> **"Which vital sign drives this alarm?"** — the question a bedside nurse will ask.


In [ ]:
try:
    import shap

    # Use a sample of test data for speed
    N_SHAP = min(500, len(X_test_sc))
    X_bg   = shap.sample(X_train_sc, 100, random_state=42)
    X_samp = X_test_sc[:N_SHAP]

    explainer   = shap.LinearExplainer(lr, X_bg, feature_perturbation="interventional")
    shap_values = explainer.shap_values(X_samp)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_samp,
                      feature_names=ALL_FEATURES,
                      max_display=15,
                      show=False)
    plt.title("SHAP Feature Importance – Logistic Regression")
    plt.tight_layout()
    plt.savefig('shap_summary.png', bbox_inches='tight')
    plt.show()

except ImportError:
    print("shap not installed – run  !pip install shap  then re-run this cell")


## Step 10 – Discussion & Reflection Questions

These questions mirror those in the project spec. Write your answers below each prompt.

---

**Q1 — Data Leakage**  
Why is splitting by *patient* rather than by *window* critical?  
What would happen to your AUROC if the same patient appeared in both train and test?

*Your answer:*

---

**Q2 — Informative Missingness**  
ICU vitals are frequently missing *not at random*. What does this mean, and how should it change your imputation strategy?

*Your answer:*

---

**Q3 — AUROC vs AUPRC**  
Your model achieves AUROC 0.85. Is that good?  
What does AUPRC add to the picture when class imbalance is ~7%?

*Your answer:*

---

**Q4 — Utility Score vs AUROC**  
The PhysioNet utility score rewards early alarms asymmetrically. How does optimising it change your decision threshold?

*Your answer:*

---

**Q5 — Clinical Deployment**  
If deployed in a real ICU, who sees the alert and what action do they take?  
What false alarm rate would nurses tolerate before ignoring the system?

*Your answer:*

---

**Q6 — Comparison to qSOFA**  
qSOFA is an established bedside sepsis screen (3 binary criteria).  
How would you benchmark your model against it fairly?

*Your answer:*

---

**🎯 Extension Challenge**  
Modify the LSTM to output a per-hour risk *trajectory* rather than a single prediction per window.  
Plot the risk trajectory for 5 sepsis patients and 5 non-sepsis patients.  
What patterns do you see?


---
## 📋 Results Summary

| Metric | Logistic Regression | LSTM |
|--------|-------------------|------|
| AUROC  | — | — |
| AUPRC  | — | — |
| Utility Score | — | — |

> Fill in your results above after running all cells.

---
*Notebook: I 320D Spring 2026 · Project 4: Sepsis Onset Prediction*  
*University of Texas at Austin · School of Information*
